In [45]:
import time
import zipfile
from pathlib import Path

import cdsapi
import numpy as np
import pandas as pd
from argopy import ArgoIndex

In [48]:
# Index of all the Argo profiles (downloaded once, then read from the local cache)
idx = ArgoIndex(index_file="core", cache=False)

# Positions and dates of the profiles of one float
wmo = 6900166
idx.query.wmo(wmo)
positions = idx.to_dataframe()[["wmo", "cyc", "date", "latitude", "longitude"]].sort_values("cyc")

print(f"Float {wmo}: {len(positions)} profiles, from {positions['date'].min()} to {positions['date'].max()}")

Float 6900166: 124 profiles, from 2002-05-16 17:21:59 to 2005-09-27 18:12:36


In [49]:
# Area covered by the float plus a margin and rounded outward on the ERA5 grid
margin = 0.5        # degrees, at least the matchup radius (0.5° is about 55 km)
grid_step = 0.25    # degrees, resolution of ERA5

if positions["longitude"].max() - positions["longitude"].min() > 180:
    raise ValueError("cross antemeridian")

west = np.floor((positions["longitude"].min() - margin) / grid_step) * grid_step
east = np.ceil((positions["longitude"].max() + margin) / grid_step) * grid_step
south = max(np.floor((positions["latitude"].min() - margin) / grid_step) * grid_step, -90)
north = min(np.ceil((positions["latitude"].max() + margin) / grid_step) * grid_step, 90)

# Period covered by the float, in whole days, plus one day before and one day after
# (the matchup looks for grid values up to one day away from each profile)
first_day = positions["date"].min().floor("D") - pd.Timedelta(days=1)
last_day = positions["date"].max().floor("D") + pd.Timedelta(days=1)
all_days = pd.date_range(first_day, last_day, freq="D")

print(f"Area: north {north}, west {west}, south {south}, east {east}")
print(f"Period: {first_day.date()} to {last_day.date()} ({len(all_days)} days)")

Area: north 40.5, west -33.75, south 35.5, east -15.25
Period: 2002-05-15 to 2005-09-28 (1233 days)


In [50]:
# What to download from ERA5
variables = [
    "mean_surface_latent_heat_flux",
    "mean_surface_sensible_heat_flux",
    "mean_surface_net_short_wave_radiation_flux",
    "mean_surface_net_long_wave_radiation_flux",
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
]
out_dir = Path(f"era5_float_{wmo}")          # one sub-folder and one csv report per test

# Hours of the day to download, by number of hours per day
HOURS = {
    1: ["12:00"],
    2: ["00:00", "12:00"],
    4: ["00:00", "06:00", "12:00", "18:00"],
    8: ["00:00", "03:00", "06:00", "09:00", "12:00", "15:00", "18:00", "21:00"],
    24: [f"{h:02d}:00" for h in range(24)],
}

# Part of the request shared by all the requests (each test adds the dates and the hours)
base_request = {
    "product_type": ["reanalysis"],
    "variable": variables,
    "area": [north, west, south, east],      # CDS expects this order
    "data_format": "netcdf",
    "download_format": "unarchived",
}

# Connection to CDS, with the key stored in ~/.cdsapirc.
# sleep_max=10: check every 10 s if a request is ready (by default it can wait up to 120 s)
client = cdsapi.Client(sleep_max=10)

# Columns of the csv report written by each test (times in seconds)
report_columns = [
    "file", "years", "months", "n_days", "n_hours",   # what was asked
    "request_id", "status", "created",                # how CDS recorded it
    "queue_s",                                        # waiting in the CDS queue
    "processing_s",                                   # work done by CDS
    "server_total_s",                                 # queue + processing
    "ready_seen_s",                                   # from sending to the moment we see the result is ready
    "download_s",                                     # download of the file only
    "client_total_s",                                 # everything seen from here, download included
    "size_MB", "error",
]


def make_request(years, months, days, hours):
    """Return the base request completed with lists of years, months, days and hours.

    CDS combines every listed year with every listed month and every listed day.
    """
    request = dict(base_request)
    request["year"] = [str(y) for y in years]
    request["month"] = [f"{m:02d}" for m in months]
    request["day"] = [f"{d:02d}" for d in days]
    request["time"] = hours
    return request


def seconds_between(start, end):
    """Seconds between two CDS timestamps (text), or None if one is missing"""
    if not start or not end:
        return None
    return round((pd.Timestamp(end) - pd.Timestamp(start)).total_seconds(), 1)


def extract_if_zip(file):
    """If CDS sent a zip archive instead of a NetCDF file, extract it and rename the archive .zip.

    CDS puts the instantaneous variables (winds) and the mean rates (fluxes) in two NetCDF files
    named ...stepType-instant.nc and ...stepType-avg.nc. They are saved next to the archive as
    <name>_instant.nc and <name>_avg.nc.
    """
    if not zipfile.is_zipfile(file):
        return
    with zipfile.ZipFile(file) as archive:
        for member in archive.namelist():
            kind = Path(member).stem.split("stepType-")[-1]    # "instant" or "avg"
            file.with_name(f"{file.stem}_{kind}.nc").write_bytes(archive.read(member))
    file.rename(file.with_suffix(".zip"))


def already_downloaded(file):
    """True if the NetCDF file is there, or the archive it was extracted from"""
    return file.exists() or file.with_suffix(".zip").exists()


def download(request, file):
    """Send one request to CDS, save the result in file, and return its row of the report"""
    row = {
        "file": file.name,
        "years": " ".join(request["year"]),
        "months": " ".join(request["month"]),
        "n_days": len(request["day"]),
        "n_hours": len(request["time"]),
    }
    file.parent.mkdir(parents=True, exist_ok=True)
    started = time.perf_counter()
    try:
        result = client.retrieve("reanalysis-era5-single-levels", request)   # returns once the result is ready
        ready_seen = time.perf_counter()
        result.download(str(file))
    except Exception as error:        # e.g. request refused because it is too large
        row["status"] = "failed"
        row["error"] = str(error)
        return row
    finished = time.perf_counter()
    row["ready_seen_s"] = round(ready_seen - started, 1)
    row["download_s"] = round(finished - ready_seen, 1)
    row["client_total_s"] = round(finished - started, 1)
    row["size_MB"] = round(file.stat().st_size / 1e6, 2)   # size of what CDS sent (zip or NetCDF)
    extract_if_zip(file)

    # Times recorded by CDS for this request
    row["request_id"] = result.url.split("/")[-2]   # the url ends with /jobs/<request id>/results
    job = client.client.get_remote(row["request_id"]).json
    row["status"] = job["status"]
    row["created"] = job.get("created")
    row["queue_s"] = seconds_between(job.get("created"), job.get("started"))
    row["processing_s"] = seconds_between(job.get("started"), job.get("finished"))
    row["server_total_s"] = seconds_between(job.get("created"), job.get("finished"))
    return row


def run(test_name, requests, send):
    """Download each (request, file) pair one after the other, or only print them if send is False.

    The report of the requests sent is written to <out_dir>/<test_name>_requests.csv,
    even if the cell is stopped before the end.
    """
    rows = []
    try:
        for request, file in requests:
            if already_downloaded(file):
                print(f"{file.name}: already downloaded, skipped")
            elif not send:
                print(f"{file.name}: years {' '.join(request['year'])}, months {' '.join(request['month'])}, "
                      f"{len(request['day'])} days, {len(request['time'])} hours")
            else:
                row = download(request, file)
                rows.append(row)
                if row["status"] == "failed":
                    print(f"{file.name}: FAILED, {row['error']}")
                else:
                    print(f"{file.name}: {row['client_total_s'] / 60:.1f} min, {row['size_MB']} MB "
                          f"(request {row['request_id']})")
    finally:
        report = pd.DataFrame(rows, columns=report_columns)
        if rows:
            report_file = out_dir / f"{test_name}_requests.csv"
            out_dir.mkdir(parents=True, exist_ok=True)
            report.to_csv(report_file, index=False)
            print(f"Report written to {report_file}")

    print(f"{test_name}: {len(requests)} requests, {report['client_total_s'].sum() / 60:.1f} min, "
          f"{report['size_MB'].sum():.1f} MB downloaded")
    return report

2026-10-15 09:41:25,324 INFO [2026-10-06T00:00:00Z] The [CDS Terms of Use](https://cds.climate.copernicus.eu/disclaimer-privacy) will be updated on 28 October 2026. Continued use of Copernicus Products and Services through the CDS after this date constitutes acceptance of the updated terms. If you do not accept the change, please [contact ECMWF Support](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202) before 27 October 2026.
 **[More information about the upcoming change.](https://forum.ecmwf.int/t/upcoming-cds-terms-of-use-change/15374)**


In [ ]:
# Full period, 24 hours a day: one single request with every year, month and day of the period
DOWNLOAD = True   # True: send the request to CDS, False: only print it

request = make_request(sorted(set(all_days.year)), range(1, 13), range(1, 32), HOURS[24])
file = out_dir / "full_period_24h" / "era5_full_period_24h.nc"
full_period_24h = run("full_period_24h", [(request, file)], DOWNLOAD)

era5_full_period_24h.nc: FAILED, 403 Client Error: Forbidden for url: https://cds.climate.copernicus.eu/api/retrieve/v1/processes/reanalysis-era5-single-levels/execution
cost limits exceeded
Your request is too large, please reduce your selection.
Report written to era5_float_6900166/full_period_24h_requests.csv
full_period_24h: 1 requests, 0.0 min, 0.0 MB downloaded


In [ ]:
# Full period, 8 hours a day: one single request with every year, month and day of the period
DOWNLOAD = True   # True: send the request to CDS, False: only print it

request = make_request(sorted(set(all_days.year)), range(1, 13), range(1, 32), HOURS[8])
file = out_dir / "full_period_8h" / "era5_full_period_8h.nc"
full_period_8h = run("full_period_8h", [(request, file)], DOWNLOAD)

era5_full_period_8h.nc: FAILED, 403 Client Error: Forbidden for url: https://cds.climate.copernicus.eu/api/retrieve/v1/processes/reanalysis-era5-single-levels/execution
cost limits exceeded
Your request is too large, please reduce your selection.
Report written to era5_float_6900166/full_period_8h_requests.csv
full_period_8h: 1 requests, 0.0 min, 0.0 MB downloaded


In [ ]:
# Full period, 4 hours a day: one single request with every year, month and day of the period
DOWNLOAD = True   # True: send the request to CDS, False: only print it

request = make_request(sorted(set(all_days.year)), range(1, 13), range(1, 32), HOURS[4])
file = out_dir / "full_period_4h" / "era5_full_period_4h.nc"
full_period_4h = run("full_period_4h", [(request, file)], DOWNLOAD)

era5_full_period_4h.nc: FAILED, 403 Client Error: Forbidden for url: https://cds.climate.copernicus.eu/api/retrieve/v1/processes/reanalysis-era5-single-levels/execution
cost limits exceeded
Your request is too large, please reduce your selection.
Report written to era5_float_6900166/full_period_4h_requests.csv
full_period_4h: 1 requests, 0.0 min, 0.0 MB downloaded


In [32]:
# Full period, 2 hours a day: one single request with every year, month and day of the period
DOWNLOAD = True   # True: send the request to CDS, False: only print it

request = make_request(sorted(set(all_days.year)), range(1, 13), range(1, 32), HOURS[2])
file = out_dir / "full_period_2h" / "era5_full_period_2h.nc"
full_period_2h = run("full_period_2h", [(request, file)], DOWNLOAD)

2026-10-14 13:07:05,585 INFO Request ID is 5c1e230d-6c6f-47a8-b9dd-1f0c82a72889
2026-10-14 13:07:05,642 INFO status has been updated to accepted
2026-10-14 13:08:07,312 INFO status has been updated to running
2026-10-14 13:24:51,817 INFO status has been updated to successful
2026-10-14 13:26:31,176 INFO Request ID is 5c1e230d-6c6f-47a8-b9dd-1f0c82a72889           


era5_full_period_2h.nc: 19.4 min, 56.84 MB (request 5c1e230d-6c6f-47a8-b9dd-1f0c82a72889)
Report written to era5_float_6900166/full_period_2h_requests.csv
full_period_2h: 1 requests, 19.4 min, 56.8 MB downloaded


In [ ]:
# Full period, 1 hour a day: one single request with every year, month and day of the period
DOWNLOAD = True   # True: send the request to CDS, False: only print it

request = make_request(sorted(set(all_days.year)), range(1, 13), range(1, 32), HOURS[1])
file = out_dir / "full_period_1h" / "era5_full_period_1h.nc"
full_period_1h = run("full_period_1h", [(request, file)], DOWNLOAD)

2026-10-14 13:52:06,235 INFO Request ID is 40383628-d707-46e9-95aa-33276f91f489
2026-10-14 13:52:06,306 INFO status has been updated to accepted


In [36]:
# Year used by the next tests (one year, 6 months, 3 months, one month):
# a complete year of the float, so that every test covers the same days
test_year = 2003

In [37]:
# One year, 24 hours a day: one single request with every month and day of test_year
DOWNLOAD = True   # True: send the request to CDS, False: only print it

request = make_request([test_year], range(1, 13), range(1, 32), HOURS[24])
file = out_dir / "year_24h" / f"era5_{test_year}_24h.nc"
year_24h = run("year_24h", [(request, file)], DOWNLOAD)

era5_2003_24h.nc: FAILED, 403 Client Error: Forbidden for url: https://cds.climate.copernicus.eu/api/retrieve/v1/processes/reanalysis-era5-single-levels/execution
cost limits exceeded
Your request is too large, please reduce your selection.
Report written to era5_float_6900166/year_24h_requests.csv
year_24h: 1 requests, 0.0 min, 0.0 MB downloaded


In [38]:
# One year, 8 hours a day: one single request with every month and day of test_year
DOWNLOAD = True   # True: send the request to CDS, False: only print it

request = make_request([test_year], range(1, 13), range(1, 32), HOURS[8])
file = out_dir / "year_8h" / f"era5_{test_year}_8h.nc"
year_8h = run("year_8h", [(request, file)], DOWNLOAD)

2026-10-14 14:11:13,039 INFO Request ID is 2edf65fe-b998-42e8-9820-b318a094032f
2026-10-14 14:11:13,110 INFO status has been updated to accepted
2026-10-14 14:12:09,594 INFO status has been updated to running
2026-10-14 14:30:27,418 INFO status has been updated to successful
2026-10-14 14:30:30,367 INFO Request ID is 2edf65fe-b998-42e8-9820-b318a094032f           


era5_2003_8h.nc: 19.3 min, 57.36 MB (request 2edf65fe-b998-42e8-9820-b318a094032f)
Report written to era5_float_6900166/year_8h_requests.csv
year_8h: 1 requests, 19.3 min, 57.4 MB downloaded


In [39]:
# One year, 4 hours a day: one single request with every month and day of test_year
DOWNLOAD = True   # True: send the request to CDS, False: only print it

request = make_request([test_year], range(1, 13), range(1, 32), HOURS[4])
file = out_dir / "year_4h" / f"era5_{test_year}_4h.nc"
year_4h = run("year_4h", [(request, file)], DOWNLOAD)

2026-10-14 14:32:21,201 INFO Request ID is 21f80ad9-1797-4501-bc2f-d9d6bcb8bad9
2026-10-14 14:32:21,294 INFO status has been updated to accepted
2026-10-14 14:35:46,146 INFO status has been updated to running
2026-10-14 14:45:09,845 INFO status has been updated to successful
2026-10-14 14:45:11,658 INFO Request ID is 21f80ad9-1797-4501-bc2f-d9d6bcb8bad9           


era5_2003_4h.nc: 13.0 min, 25.97 MB (request 21f80ad9-1797-4501-bc2f-d9d6bcb8bad9)
Report written to era5_float_6900166/year_4h_requests.csv
year_4h: 1 requests, 13.0 min, 26.0 MB downloaded


In [40]:
# One year, 2 hours a day: one single request with every month and day of test_year
DOWNLOAD = True   # True: send the request to CDS, False: only print it

request = make_request([test_year], range(1, 13), range(1, 32), HOURS[2])
file = out_dir / "year_2h" / f"era5_{test_year}_2h.nc"
year_2h = run("year_2h", [(request, file)], DOWNLOAD)

2026-10-14 15:06:25,161 INFO Request ID is 2ceb50a0-8fea-483a-939c-e844e495a8d4
2026-10-14 15:06:25,237 INFO status has been updated to accepted
2026-10-14 15:07:58,321 INFO status has been updated to running
2026-10-14 15:12:28,530 INFO status has been updated to successful
2026-10-14 15:12:33,130 INFO Request ID is 2ceb50a0-8fea-483a-939c-e844e495a8d4           


era5_2003_2h.nc: 6.1 min, 13.06 MB (request 2ceb50a0-8fea-483a-939c-e844e495a8d4)
Report written to era5_float_6900166/year_2h_requests.csv
year_2h: 1 requests, 6.1 min, 13.1 MB downloaded


In [41]:
# One year, 1 hour a day: one single request with every month and day of test_year
DOWNLOAD = True   # True: send the request to CDS, False: only print it

request = make_request([test_year], range(1, 13), range(1, 32), HOURS[1])
file = out_dir / "year_1h" / f"era5_{test_year}_1h.nc"
year_1h = run("year_1h", [(request, file)], DOWNLOAD)

2026-10-14 15:24:46,716 INFO Request ID is 29d9a1c8-06c4-4435-b9c8-bec4fe588fe0
2026-10-14 15:24:46,785 INFO status has been updated to accepted
2026-10-14 15:25:09,641 INFO status has been updated to running
2026-10-14 15:28:07,299 INFO status has been updated to successful
2026-10-14 15:28:08,919 INFO Request ID is 29d9a1c8-06c4-4435-b9c8-bec4fe588fe0           


era5_2003_1h.nc: 3.4 min, 7.1 MB (request 29d9a1c8-06c4-4435-b9c8-bec4fe588fe0)
Report written to era5_float_6900166/year_1h_requests.csv
year_1h: 1 requests, 3.4 min, 7.1 MB downloaded


In [ ]:
# 6 months, 24 hours a day: two requests to cover test_year (January-June, then July-December)
DOWNLOAD = True   # True: send the requests to CDS, False: only print them

requests_to_send = []
for first_month in [1, 7]:
    months = range(first_month, first_month + 6)
    request = make_request([test_year], months, range(1, 32), HOURS[24])
    file = out_dir / "6months_24h" / f"era5_{test_year}{first_month:02d}_6months_24h.nc"
    requests_to_send.append((request, file))

six_months_24h = run("6months_24h", requests_to_send, DOWNLOAD)

In [ ]:
# 6 months, 8 hours a day: two requests to cover test_year (January-June, then July-December)
DOWNLOAD = True   # True: send the requests to CDS, False: only print them

requests_to_send = []
for first_month in [1, 7]:
    months = range(first_month, first_month + 6)
    request = make_request([test_year], months, range(1, 32), HOURS[8])
    file = out_dir / "6months_8h" / f"era5_{test_year}{first_month:02d}_6months_8h.nc"
    requests_to_send.append((request, file))

six_months_8h = run("6months_8h", requests_to_send, DOWNLOAD)

In [ ]:
# 6 months, 4 hours a day: two requests to cover test_year (January-June, then July-December)
DOWNLOAD = True   # True: send the requests to CDS, False: only print them

requests_to_send = []
for first_month in [1, 7]:
    months = range(first_month, first_month + 6)
    request = make_request([test_year], months, range(1, 32), HOURS[4])
    file = out_dir / "6months_4h" / f"era5_{test_year}{first_month:02d}_6months_4h.nc"
    requests_to_send.append((request, file))

six_months_4h = run("6months_4h", requests_to_send, DOWNLOAD)

In [ ]:
# 6 months, 2 hours a day: two requests to cover test_year (January-June, then July-December)
DOWNLOAD = True   # True: send the requests to CDS, False: only print them

requests_to_send = []
for first_month in [1, 7]:
    months = range(first_month, first_month + 6)
    request = make_request([test_year], months, range(1, 32), HOURS[2])
    file = out_dir / "6months_2h" / f"era5_{test_year}{first_month:02d}_6months_2h.nc"
    requests_to_send.append((request, file))

six_months_2h = run("6months_2h", requests_to_send, DOWNLOAD)

In [ ]:
# 6 months, 1 hour a day: two requests to cover test_year (January-June, then July-December)
DOWNLOAD = True   # True: send the requests to CDS, False: only print them

requests_to_send = []
for first_month in [1, 7]:
    months = range(first_month, first_month + 6)
    request = make_request([test_year], months, range(1, 32), HOURS[1])
    file = out_dir / "6months_1h" / f"era5_{test_year}{first_month:02d}_6months_1h.nc"
    requests_to_send.append((request, file))

six_months_1h = run("6months_1h", requests_to_send, DOWNLOAD)

In [ ]:
# 3 months, 24 hours a day: four requests to cover test_year, one per quarter
DOWNLOAD = True   # True: send the requests to CDS, False: only print them

requests_to_send = []
for first_month in [1, 4, 7, 10]:
    months = range(first_month, first_month + 3)
    request = make_request([test_year], months, range(1, 32), HOURS[24])
    file = out_dir / "3months_24h" / f"era5_{test_year}{first_month:02d}_3months_24h.nc"
    requests_to_send.append((request, file))

three_months_24h = run("3months_24h", requests_to_send, DOWNLOAD)

In [ ]:
# 3 months, 8 hours a day: four requests to cover test_year, one per quarter
DOWNLOAD = True   # True: send the requests to CDS, False: only print them

requests_to_send = []
for first_month in [1, 4, 7, 10]:
    months = range(first_month, first_month + 3)
    request = make_request([test_year], months, range(1, 32), HOURS[8])
    file = out_dir / "3months_8h" / f"era5_{test_year}{first_month:02d}_3months_8h.nc"
    requests_to_send.append((request, file))

three_months_8h = run("3months_8h", requests_to_send, DOWNLOAD)

In [ ]:
# 3 months, 4 hours a day: four requests to cover test_year, one per quarter
DOWNLOAD = True   # True: send the requests to CDS, False: only print them

requests_to_send = []
for first_month in [1, 4, 7, 10]:
    months = range(first_month, first_month + 3)
    request = make_request([test_year], months, range(1, 32), HOURS[4])
    file = out_dir / "3months_4h" / f"era5_{test_year}{first_month:02d}_3months_4h.nc"
    requests_to_send.append((request, file))

three_months_4h = run("3months_4h", requests_to_send, DOWNLOAD)

In [ ]:
# 3 months, 2 hours a day: four requests to cover test_year, one per quarter
DOWNLOAD = True   # True: send the requests to CDS, False: only print them

requests_to_send = []
for first_month in [1, 4, 7, 10]:
    months = range(first_month, first_month + 3)
    request = make_request([test_year], months, range(1, 32), HOURS[2])
    file = out_dir / "3months_2h" / f"era5_{test_year}{first_month:02d}_3months_2h.nc"
    requests_to_send.append((request, file))

three_months_2h = run("3months_2h", requests_to_send, DOWNLOAD)

In [ ]:
# 3 months, 1 hour a day: four requests to cover test_year, one per quarter
DOWNLOAD = True   # True: send the requests to CDS, False: only print them

requests_to_send = []
for first_month in [1, 4, 7, 10]:
    months = range(first_month, first_month + 3)
    request = make_request([test_year], months, range(1, 32), HOURS[1])
    file = out_dir / "3months_1h" / f"era5_{test_year}{first_month:02d}_3months_1h.nc"
    requests_to_send.append((request, file))

three_months_1h = run("3months_1h", requests_to_send, DOWNLOAD)

In [ ]:
# 1 month, 24 hours a day: twelve requests to cover test_year, one per month
DOWNLOAD = True   # True: send the requests to CDS, False: only print them

requests_to_send = []
for month in range(1, 13):
    request = make_request([test_year], [month], range(1, 32), HOURS[24])
    file = out_dir / "1month_24h" / f"era5_{test_year}{month:02d}_1month_24h.nc"
    requests_to_send.append((request, file))

one_month_24h = run("1month_24h", requests_to_send, DOWNLOAD)

In [ ]:
# 1 month, 8 hours a day: twelve requests to cover test_year, one per month
DOWNLOAD = True   # True: send the requests to CDS, False: only print them

requests_to_send = []
for month in range(1, 13):
    request = make_request([test_year], [month], range(1, 32), HOURS[8])
    file = out_dir / "1month_8h" / f"era5_{test_year}{month:02d}_1month_8h.nc"
    requests_to_send.append((request, file))

one_month_8h = run("1month_8h", requests_to_send, DOWNLOAD)

In [ ]:
# 1 month, 4 hours a day: twelve requests to cover test_year, one per month
DOWNLOAD = True   # True: send the requests to CDS, False: only print them

requests_to_send = []
for month in range(1, 13):
    request = make_request([test_year], [month], range(1, 32), HOURS[4])
    file = out_dir / "1month_4h" / f"era5_{test_year}{month:02d}_1month_4h.nc"
    requests_to_send.append((request, file))

one_month_4h = run("1month_4h", requests_to_send, DOWNLOAD)

In [ ]:
# 1 month, 2 hours a day: twelve requests to cover test_year, one per month
DOWNLOAD = True   # True: send the requests to CDS, False: only print them

requests_to_send = []
for month in range(1, 13):
    request = make_request([test_year], [month], range(1, 32), HOURS[2])
    file = out_dir / "1month_2h" / f"era5_{test_year}{month:02d}_1month_2h.nc"
    requests_to_send.append((request, file))

one_month_2h = run("1month_2h", requests_to_send, DOWNLOAD)

In [51]:
# 1 month, 1 hour a day: twelve requests to cover test_year, one per month
DOWNLOAD = True   # True: send the requests to CDS, False: only print them

requests_to_send = []
for month in range(1, 13):
    request = make_request([test_year], [month], range(1, 32), HOURS[1])
    file = out_dir / "1month_1h" / f"era5_{test_year}{month:02d}_1month_1h.nc"
    requests_to_send.append((request, file))

one_month_1h = run("1month_1h", requests_to_send, DOWNLOAD)

2026-10-15 09:41:36,437 INFO Request ID is de32dcbf-2009-4cac-abe4-612d03c54568
2026-10-15 09:41:36,514 INFO status has been updated to accepted
2026-10-15 11:01:42,660 INFO status has been updated to running
2026-10-15 11:01:52,727 INFO status has been updated to successful
2026-10-15 11:01:53,780 INFO Request ID is de32dcbf-2009-4cac-abe4-612d03c54568         


era5_200301_1month_1h.nc: 80.3 min, 0.7 MB (request de32dcbf-2009-4cac-abe4-612d03c54568)


2026-10-15 11:01:54,333 INFO Request ID is 6abde638-81c0-43c4-8b85-77db70e5d817
2026-10-15 11:01:54,395 INFO status has been updated to accepted
2026-10-15 12:05:07,745 INFO status has been updated to running
2026-10-15 12:05:58,163 INFO status has been updated to successful
2026-10-15 12:05:59,143 INFO Request ID is 6abde638-81c0-43c4-8b85-77db70e5d817         


era5_200302_1month_1h.nc: 64.1 min, 0.64 MB (request 6abde638-81c0-43c4-8b85-77db70e5d817)


2026-10-15 12:06:01,411 INFO Request ID is 7b18d152-5676-436e-af83-3f84a74bfc13
2026-10-15 12:06:01,478 INFO status has been updated to accepted
2026-10-15 13:25:00,934 INFO status has been updated to running
2026-10-15 13:25:22,414 INFO status has been updated to successful
2026-10-15 13:25:23,445 INFO Request ID is 7b18d152-5676-436e-af83-3f84a74bfc13         


era5_200303_1month_1h.nc: 79.4 min, 0.71 MB (request 7b18d152-5676-436e-af83-3f84a74bfc13)


2026-10-15 13:25:23,944 INFO Request ID is 8c697a8d-791d-4abe-ac4f-2a7ec0d9a514
2026-10-15 13:25:24,000 INFO status has been updated to accepted


Report written to era5_float_6900166/1month_1h_requests.csv


KeyboardInterrupt: 

In [43]:
import cdsapi

dataset = "reanalysis-era5-single-levels"
request = {
    "product_type": ["reanalysis"],
    "variable": [
        "10m_u_component_of_wind",
        "10m_v_component_of_wind",
        "mean_surface_latent_heat_flux",
        "mean_surface_sensible_heat_flux",
        "mean_top_net_long_wave_radiation_flux",
        "mean_top_net_short_wave_radiation_flux"
    ],
    "year": ["2003"],
    "month": ["04"],
    "day": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12",
        "13", "14", "15",
        "16", "17", "18",
        "19", "20", "21",
        "22", "23", "24",
        "25", "26", "27",
        "28", "29", "30"
    ],
    "time": [
        "00:00", "01:00", "02:00",
        "03:00", "04:00", "05:00",
        "06:00", "07:00", "08:00",
        "09:00", "10:00", "11:00",
        "12:00", "13:00", "14:00",
        "15:00", "16:00", "17:00",
        "18:00", "19:00", "20:00",
        "21:00", "22:00", "23:00"
    ],
    "data_format": "netcdf",
    "download_format": "unarchived",
    "area": [41, -35, 35, -15]
}

client = cdsapi.Client()
client.retrieve(dataset, request).download()

2026-10-15 07:55:02,986 INFO [2026-10-06T00:00:00Z] The [CDS Terms of Use](https://cds.climate.copernicus.eu/disclaimer-privacy) will be updated on 28 October 2026. Continued use of Copernicus Products and Services through the CDS after this date constitutes acceptance of the updated terms. If you do not accept the change, please [contact ECMWF Support](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202) before 27 October 2026.
 **[More information about the upcoming change.](https://forum.ecmwf.int/t/upcoming-cds-terms-of-use-change/15374)**
2026-10-15 07:55:03,391 INFO Request ID is 5f265c78-cdc0-4f16-b6e6-c9562bfa757a
2026-10-15 07:55:03,469 INFO status has been updated to accepted
2026-10-15 09:17:46,418 INFO status has been updated to running
2026-10-15 09:23:47,653 INFO status has been updated to successful


'83faaed299292ac044a0013e4a3bdc4a.zip'